In [10]:
import pandas as pd

In [11]:
# read payment csv files 
df_payment = pd.read_csv("payment_report.csv")
print(df_payment)

    report_month payment_group  product_id  source_id       volume
0        2023-01       payment          12         45    624110375
1        2023-01       payment          17         45    335715113
2        2023-01       payment          18         45    737784466
3        2023-01       payment          19         45    120963069
4        2023-01       payment          20         45    319653158
..           ...           ...         ...        ...          ...
914      2023-04       payment       15067         45      1504000
915      2023-04        refund        1976         37   3542271587
916      2023-04        refund        1976         38  13831708189
917      2023-04        refund        1976         39   1905435543
918      2023-04        refund        1976         39   3679922071

[919 rows x 5 columns]


In [12]:
# read product csv files 
df_product = pd.read_csv("product.csv")
print(df_product)


     product_id category team_own
0            17  PXXXXXB      ASD
1            18  PXXXXXB      ASD
2            20  PXXXXXB      ASD
3           287  PXXXXXB      ASD
4           372  PXXXXXB      ASD
..          ...      ...      ...
487         321  PXXXXXV      ASD
488         322  PXXXXXV      ASD
489         341  PXXXXXV      ASD
490         342  PXXXXXV      ASD
491         367  PXXXXXV      ASD

[492 rows x 3 columns]


In [13]:
# read transactions csv file
df_transactions = pd.read_csv("transactions.csv")
print(df_transactions.head(5))

   transaction_id  merchant_id   volume  transType  transStatus   sender_id  \
0      3002692434            5   100000         24            1  10199794.0   
1      3002692437          305    20000          2            1  14022211.0   
2      3001960110         7255    48605         22            1         NaN   
3      3002680710         2270  1500000          2            1  10059206.0   
4      3002680713         2275    90000          2            1  10004711.0   

   receiver_id extra_info      timeStamp  
0     199794.0        NaN  1682932054455  
1   14022211.0        NaN  1682932054912  
2   10530940.0        NaN  1682932055000  
3      59206.0        NaN  1682932055622  
4       4711.0        NaN  1682932056197  


In [14]:
# merge payment and product 
payment_product = df_payment.merge( df_product, on="product_id", how = "left" )
print(payment_product)

    report_month payment_group  product_id  source_id       volume  category  \
0        2023-01       payment          12         45    624110375   PXXXXXT   
1        2023-01       payment          17         45    335715113   PXXXXXB   
2        2023-01       payment          18         45    737784466   PXXXXXB   
3        2023-01       payment          19         45    120963069  PXXXXXM2   
4        2023-01       payment          20         45    319653158   PXXXXXB   
..           ...           ...         ...        ...          ...       ...   
914      2023-04       payment       15067         45      1504000   PXXXXXR   
915      2023-04        refund        1976         37   3542271587       NaN   
916      2023-04        refund        1976         38  13831708189       NaN   
917      2023-04        refund        1976         39   1905435543       NaN   
918      2023-04        refund        1976         39   3679922071       NaN   

    team_own  
0        ASD  
1        

Part I: EDA - Explore Data Analysis


In [15]:
# checking after merging
print(df_payment.shape)
print(payment_product.shape)
print(df_transactions.shape)

(919, 5)
(919, 7)
(1324002, 9)


In [16]:
# cheking missing value
payment_product.isna().sum()


report_month      0
payment_group     0
product_id        0
source_id         0
volume            0
category         22
team_own         22
dtype: int64

In [17]:
df_transactions.isna().sum()

transaction_id          0
merchant_id             0
volume                  0
transType               0
transStatus             0
sender_id           49059
receiver_id        164795
extra_info        1317907
timeStamp               0
dtype: int64

In [18]:
# checking duplicated payment
key_cols = ["report_month", "payment_group", "product_id", "source_id"]

duplicates = payment_product.duplicated(subset=key_cols).sum()

print("Number of duplicates:", duplicates)

Number of duplicates: 5


In [19]:
# Dropping 5 duplicates
after_dr_payment_product = payment_product.drop_duplicates(subset= key_cols)


In [20]:
# checking duplicated transactions
print("Duplicated transaction_id:", df_transactions["transaction_id"].duplicated().sum())

Duplicated transaction_id: 28


In [21]:
# checking incorrect data types
payment_product.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 919 entries, 0 to 918
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   report_month   919 non-null    object
 1   payment_group  919 non-null    object
 2   product_id     919 non-null    int64 
 3   source_id      919 non-null    int64 
 4   volume         919 non-null    int64 
 5   category       897 non-null    object
 6   team_own       897 non-null    object
dtypes: int64(3), object(4)
memory usage: 50.4+ KB


In [22]:
# Convert data type of 'report_month' to datetime
payment_product["report_month"] = pd.to_datetime(payment_product["report_month"])

In [23]:
df_transactions.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1324002 entries, 0 to 1324001
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   transaction_id  1324002 non-null  int64  
 1   merchant_id     1324002 non-null  int64  
 2   volume          1324002 non-null  int64  
 3   transType       1324002 non-null  int64  
 4   transStatus     1324002 non-null  int64  
 5   sender_id       1274943 non-null  float64
 6   receiver_id     1159207 non-null  float64
 7   extra_info      6095 non-null     object 
 8   timeStamp       1324002 non-null  int64  
dtypes: float64(2), int64(6), object(1)
memory usage: 90.9+ MB


In [24]:
# Summary statistics
print(payment_product.describe())

                        report_month    product_id   source_id        volume
count                            919    919.000000  919.000000  9.190000e+02
mean   2023-02-19 06:05:05.549510400   1192.517954   44.875952  1.978574e+08
min              2023-01-01 00:00:00      3.000000   37.000000  5.500000e+03
25%              2023-02-01 00:00:00    640.000000   45.000000  1.250000e+06
50%              2023-03-01 00:00:00   1059.000000   45.000000  7.982015e+06
75%              2023-04-01 00:00:00   1585.000000   45.000000  5.447599e+07
max              2023-04-01 00:00:00  15067.000000   45.000000  1.383171e+10
std                              NaN   1293.463329    0.910995  8.367595e+08


In [25]:
print(df_transactions.describe())

       transaction_id   merchant_id        volume     transType   transStatus  \
count    1.324002e+06  1.324002e+06  1.324002e+06  1.324002e+06  1.324002e+06   
mean     3.002233e+09  2.460318e+03  2.388059e+05  6.979222e+00 -1.204625e+01   
std      1.042606e+07  3.304277e+03  9.681009e+05  7.459714e+00  5.577823e+01   
min      3.000000e+09  5.000000e+00  1.000000e+00  2.000000e+00 -1.333000e+03   
25%      3.001121e+09  3.050000e+02  1.000000e+04  2.000000e+00  1.000000e+00   
50%      3.002200e+09  2.250000e+03  3.000000e+04  2.000000e+00  1.000000e+00   
75%      3.003255e+09  2.270000e+03  1.000000e+05  8.000000e+00  1.000000e+00   
max      6.000066e+09  1.625250e+05  7.869148e+07  3.000000e+01  1.000000e+00   

          sender_id   receiver_id     timeStamp  
count  1.274943e+06  1.159207e+06  1.324002e+06  
mean   1.033938e+08  2.084884e+08  1.683130e+12  
std    6.234305e+08  9.287794e+08  1.707815e+08  
min    1.000001e+07 -6.300000e+01  1.682874e+12  
25%    1.005657e+07 

In [28]:
# check incorrect values in 'payment_product'
print(payment_product[payment_product["volume"] <= 0])



Empty DataFrame
Columns: [report_month, payment_group, product_id, source_id, volume, category, team_own]
Index: []


In [29]:
# check incorrect values in 'transactions'
print(df_transactions[df_transactions["volume"] <= 0])


Empty DataFrame
Columns: [transaction_id, merchant_id, volume, transType, transStatus, sender_id, receiver_id, extra_info, timeStamp]
Index: []
